In [1]:
# SET UP AND INITIALIZE TOOLS

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import trange

torch.manual_seed(42)

DEVICE = "cpu"
#DEVICE = "cuda"
#DEVICE = "mps"

DATA = open("shakespeare.txt", "r").read()
VOCAB = sorted(set(DATA))
VOCAB_SIZE = len(VOCAB)
VOCAB_MAP = {c:i for i,c in enumerate(VOCAB)}

def encode(s):
    return torch.tensor([VOCAB_MAP[c] for c in s])

def decode(l):
    return ''.join([VOCAB[i] for i in l])


DATASET_TRAINING = encode(DATA[:int(len(DATA)*0.9)]).to(DEVICE)
DATASET_TESTING = encode(DATA[int(len(DATA)*0.9):]).to(DEVICE)


def get_batch(input_set, batch_size, block_size):
    ix = torch.randint(0, len(input_set) - block_size, (batch_size, 1), device = input_set.device)
    ix = ix + torch.arange(block_size+1, device = input_set.device)
    batch = input_set[ix]
    return batch[...,:-1], batch[...,1:]

@torch.no_grad()
def estimate_losses(model):
    model.eval()
    losses = [ 
        torch.stack([
            model(*get_batch(dataset, 32, model.context_window)) for _ in range(100)
        ]).mean() 
        for dataset in [DATASET_TESTING, DATASET_TRAINING]]
    model.train()
    return losses[0], losses[1]

#get_batch(DATASET_TESTING, 4,8)


In [ ]:

class MultiHeadAttention(nn.Module):
    def __init__(self, embedding_dim, head_size, dropout_p):
        super().__init__()
        assert embedding_dim % head_size == 0, "Error embedding_dim is not divisible by head_size"
        self.heads = embedding_dim // head_size
        self.head_size = head_size 
        self.dropout = nn.Dropout(dropout_p)
        self.qkv = nn.Linear(embedding_dim, 3 * embedding_dim, bias = False)
        self.proj = nn.Linear(embedding_dim, embedding_dim)
        
    def forward(self, x):
        # input: [B,W,E]
        B,W,E = x.shape
        H = self.heads 
        A = self.head_size
        qkv = self.qkv(x)  # [B, W, 3E]
        q,k,v = qkv.view(B,W,3,H,A).permute(2,0,1,3,4).squeeze(0) # [B,W,H,A]
        att = F.softmax(q @ k.transpose(-2,-1), dim=-1)
        att = self.dropout(att)
        out = (att @ v).view(B,W,E)
        return self.proj(out)

class FeedForward(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        self.lin = nn.Linear(embedding_dim, 4 * embedding_dim)
        self.gelu = nn.GELU()
        self.proj = nn.Linear(embedding_dim * 4, embedding_dim)
        
    def forward(self, x):
        return self.proj(self.gelu(self.lin(x)))

class AttentionBlock(nn.Module):
    def __init__(self, embedding_dim, head_size, dropout_p):
        super().__init__()
        self.dropout = nn.Dropout(dropout_p)
        self.lnorm1 = nn.LayerNorm(embedding_dim)
        self.lnorm2 = nn.LayerNorm(embedding_dim)
        self.mha = MultiHeadAttention(embedding_dim, head_size, dropout_p)
        self.ff = FeedForward(embedding_dim)

    def forward(self, x):
        x = x + self.dropout(self.mha(self.lnorm1(x)))
        x = x + self.dropout(self.ff(self.lnorm1(x)))
        return x

class Transformer(nn.Module):
    def __init__(self, vocabulary_size, context_window, embedding_dim, attention_blocks, head_size, dropout_p):
        super().__init__()
        self.vocabulary_size = vocabulary_size
        self.context_window = context_window
        self.embedding_token = nn.Embedding(vocabulary_size, embedding_dim)
        self.embedding_posit = nn.Embedding(context_window, embedding_dim)
        self.att = nn.Sequential(*[AttentionBlock(embedding_dim, head_size, dropout_p) for _ in range(attention_blocks)])
        self.dropout = nn.Dropout(dropout_p)
        self.to_logits = nn.Linear(embedding_dim, vocabulary_size)
        self.register_buffer("positions", torch.arange(0, context_window))

    def forward(self, x, y = None):
        x = self.embedding_token(x) + self.embedding_posit(self.positions)
        x = self.dropout(x)
        x = self.att(x)
        x = self.to_logits(x)
        if y!=None:
            lg = x.reshape(-1,self.vocabulary_size)
            return F.cross_entropy(lg, y.contiguous().view(-1)).mean()
        else:
            return x

    @torch.no_grad()
    def generate(self, num_of_chars, prompt_encoded = torch.tensor([])):
        self.eval()
        padding = max(self.context_window - len(prompt_encoded), 0)
        out = ([0]*padding) + prompt_encoded.tolist()
        for _ in range(num_of_chars):
            inp = torch.tensor(out[-self.context_window:]).view(1,-1)
            l = self(inp)
            r = F.softmax(l, dim=-1)
            out.append(torch.multinomial(r[0,-1], num_samples=1).item())
        self.train()
        return out[padding:]
           

m = Transformer(VOCAB_SIZE, context_window=32, embedding_dim=64, attention_blocks=6, head_size=16, dropout_p=0.1)

o = optim.AdamW(m.parameters(), lr=3e4)

def training_loop(model, optimizer, iterations):
    print(f"Model has {sum(p.numel() for p in model.parameters()):,} parameters.")
    pbar = trange(iterations, desc="Training")
    for i in pbar:
        optimizer.zero_grad()
        loss = m(*get_batch(DATASET_TRAINING, 32, m.context_window))
        loss.backward()
        optimizer.step()
        #if i%100==99:
        if True:
            ltest,ltrain = estimate_losses(m)
            pbar.set_postfix({
                "test_loss": f"{ltest:.4f}",
                "tr_loss": f"{ltrain:.4f}"})
    losses = estimate_losses(m)
    return f"Test loss = {losses[0]:.4f}, Train loss = Test loss = {losses[1]:.4f}"
    
training_loop(m, o, 100)

print(decode(m.generate(100, prompt_encoded = encode("Hello "))))



#MultiHeadAttention(8, 4, 0.1)(torch.randn((4,10,8)))





#self.embeddings_token = nn.Embedding(vocabulary_size, embedding_dim)
#self.embeddings_pos = nn.Embedding(context_window, embedding_dim)        


Model has 309,185 parameters.


Training:   0%|          | 0/100 [00:00<?, ?it/s]

Training:  10%|█         | 10/100 [00:12<01:48,  1.20s/it, test_loss=nan, tr_loss=nan]                                                 


KeyboardInterrupt: 